In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import os
import warnings
from tqdm import tqdm
import pickle

In [2]:
mapping = {'albite':1,
           'an':8,
           'aug':6,
           'bt':3,
           'cal':10,
           'en':7,
           'fo':5,
           'foalb':14,
           'foaug':13,
           'gyp':11,
           'hb':2,
           'mc':9,
           'ms':4,
           'qtz':0,
           'qtzalb':12,
           'Unknown':15}

rev_mapping = {v:k for k,v in mapping.items()}

In [3]:
#Saving the names of all the csv files in the current directory
all_files = os.listdir()
test_csv_files = [x for x in all_files if x.endswith(".csv") and "test" in x]

print(f"The number of test files are {len(test_csv_files)}")

The number of test files are 7


In [4]:
#Key is the name of the csv file and the value is the values in the file
df_mapping = {}
for csv in test_csv_files:
    warnings.filterwarnings("ignore")
    df = pd.read_csv(os.path.join(csv), header=None) #Reading the value of the csv file

    x_axis = (np.array(df))[0,1:-1].astype("float64") #The wavenumbers 
    X = (np.array(df))[1:,1:-1].astype("float64") #The spectra
    y = (np.array(df))[1:,-1].astype("int64") #The class labels

    df_mapping[csv.split(".")[0]] = [x_axis,X,y]
    print(f"{csv} has {X.shape[1]} frequency components and {X.shape[0]} elements")

granite50dust_test_015s_10000.csv has 2064 frequency components and 10000 elements
granite0dust_test_030s_1373_final.csv has 2064 frequency components and 1373 elements
granite0dust_test_015s_5571_final.csv has 2064 frequency components and 5571 elements
gabbro50dust_test_015s_9740.csv has 1371 frequency components and 9740 elements
granite0dust_test_015s_4084_final.csv has 2184 frequency components and 4084 elements
gabbro0dust_test_015s_8906_final.csv has 1371 frequency components and 8906 elements
gabbro0dust_test_015s_46_final.csv has 1371 frequency components and 46 elements


In [6]:
max_freq = 1100.46
min_freq = 141.022

print(f"\n The highest minimum frequency is {min_freq}Hz and lowest maximum frequency is {max_freq}Hz")


 The highest minimum frequency is 141.022Hz and lowest maximum frequency is 1100.46Hz


In [7]:
#interpolating to new spectral region
new_df_mapping = {}
new_x_axis = np.linspace(min_freq,max_freq,1024) #of shape 1024
for key,(x_axis,x,y) in df_mapping.items():
    new_vals = []
    for i in range(x.shape[0]):
        z = np.interp(new_x_axis,x_axis,x[i])[np.newaxis, ...] #of shape (1,1024)
        new_vals.append(z)    
    
    new_vals = np.stack(new_vals) 
    new_df_mapping[key] = [new_vals,y]
    print(f"{key} has {new_vals.shape[-1]} frequency components and {new_vals.shape[0]} elements")

granite50dust_test_015s_10000 has 1024 frequency components and 10000 elements
granite0dust_test_030s_1373_final has 1024 frequency components and 1373 elements
granite0dust_test_015s_5571_final has 1024 frequency components and 5571 elements
gabbro50dust_test_015s_9740 has 1024 frequency components and 9740 elements
granite0dust_test_015s_4084_final has 1024 frequency components and 4084 elements
gabbro0dust_test_015s_8906_final has 1024 frequency components and 8906 elements
gabbro0dust_test_015s_46_final has 1024 frequency components and 46 elements


In [9]:
gn_0_spectra = []
gn_0_labels = []

gn_50_spectra = []
gn_50_labels = []

gb_0_spectra = []
gb_0_labels = []

gb_50_spectra = []
gb_50_labels = []

for key,(x,y) in new_df_mapping.items():
    if "granite0dust" in key:
        gn_0_spectra.append(x)
        gn_0_labels.append(y)
    
    elif "granite50dust" in key:
        gn_50_spectra.append(x)
        gn_50_labels.append(y)

    elif "gabbro0dust" in key:
        gb_0_spectra.append(x)
        gb_0_labels.append(y)
    
    elif "gabbro50dust" in key:
        gb_50_spectra.append(x)
        gb_50_labels.append(y)

gn_0_spectra = np.vstack(gn_0_spectra)
gn_50_spectra = np.vstack(gn_50_spectra)

gn_0_labels = np.concatenate(gn_0_labels)
gn_50_labels = np.concatenate(gn_50_labels)

gb_0_spectra = np.vstack(gb_0_spectra)
gb_50_spectra = np.vstack(gb_50_spectra)

gb_0_labels = np.concatenate(gb_0_labels)
gb_50_labels = np.concatenate(gb_50_labels)

In [10]:
with open("MLROD_test_granite_0.pkl","wb") as file:
    pickle.dump([gn_0_labels,gn_0_spectra],file) 

with open("MLROD_test_granite_50.pkl","wb") as file:
    pickle.dump([gn_50_labels,gn_50_spectra],file) 


In [11]:
with open("MLROD_test_gabbro_0.pkl","wb") as file:
    pickle.dump([gb_0_labels,gb_0_spectra],file) 

with open("MLROD_test_gabbro_50.pkl","wb") as file:
    pickle.dump([gb_50_labels,gb_50_spectra],file) 